# Doing the Same Math to a Thousand Spectra at Once
### 2.1 — NumPy Arrays: Math on Whole Spectra at Once

You ran a tray of samples overnight and exported a thousand spectra. Every one
needs the same instrument baseline removed before you can do anything else.
Correcting them one at a time — or worse, in the vendor software, spectrum by
spectrum — is not analysis, it's data entry.

A NumPy **array** lets you write the correction once and apply it to every point
of every spectrum in a single line. This is the tool the rest of the course runs
on: baseline subtraction, scatter correction, and averaging are all just
arithmetic on arrays.

> **One idea to hold onto:** a spectrum is a 1-D array; a stack of spectra is a
> 2-D array (one row per spectrum, one column per wavelength). Write the math
> once and NumPy applies it to every element — no loops.

**By the end of this notebook you will be able to:**

1. Do arithmetic on a whole spectrum at once, instead of point by point.
2. Stack many spectra into a 2-D array and baseline-correct all of them in one line.
3. Average replicate spectra and summarize a batch with array reductions.

## 1. One spectrum, whole-array arithmetic

We build a single absorbance spectrum: one band on a sloping instrument baseline.
Then we subtract a baseline from **every** point at once — no loop in sight.

In [ ]:
import numpy as np

# A wavelength axis (nm) and one absorbance band sitting on a sloping baseline.
wl = np.linspace(400, 700, 301)                       # 301 points, 1 nm apart
band = 0.9 * np.exp(-((wl - 550) ** 2) / (2 * 15 ** 2))   # a Gaussian band at 550 nm
instrument_baseline = 0.10 + 0.0005 * (wl - 400)      # a gentle sloping offset
absorbance = band + instrument_baseline

# Subtract a SINGLE number from every point -- one operation, all 301 points.
flat_corrected = absorbance - 0.10

# Subtract a whole baseline ARRAY, point-for-point (element-wise).
corrected = absorbance - instrument_baseline

print("absorbance shape:", absorbance.shape)
print("peak before:", round(absorbance.max(), 3), "AU  ->  after:", round(corrected.max(), 3), "AU")

`absorbance - instrument_baseline` lined the two arrays up by position and
subtracted them element by element. A plain Python list can't do that; an array
can, and that is the whole reason we reach for NumPy.

## 2. A stack of spectra is a 2-D array

Now the real situation: a thousand spectra from an overnight tray. We hold them in
one 2-D array of shape `(n_spectra, n_wavelengths)` — each **row** is a spectrum,
each **column** is a wavelength. (We generate them here only so the lesson is
self-contained; in practice they'd be loaded from files.)

In [ ]:
rng = np.random.default_rng(0)        # fixed seed -> identical every run
n_spectra = 1000

# Each spectrum = the same band + baseline, plus a small per-spectrum vertical
# offset and random noise. Broadcasting builds all 1000 at once:
#   band (301,) + baseline (301,) + offsets (1000,1) + noise (1000,301) -> (1000,301)
offsets = rng.normal(0.0, 0.02, size=(n_spectra, 1))
noise = rng.normal(0.0, 0.01, size=(n_spectra, wl.size))
stack = band + instrument_baseline + offsets + noise

print("stack shape:", stack.shape, "(rows = spectra, columns = wavelengths)")
print("one spectrum is a row:", stack[0].shape)
print("one wavelength across all spectra is a column:", stack[:, 0].shape)

## 3. Baseline-correct all 1000 in one line

Here is the payoff. To remove the instrument baseline from every spectrum, we
subtract the baseline **vector** from the whole **stack**. NumPy *broadcasts* the
301-length baseline across all 1000 rows automatically.

In [ ]:
# (1000, 301) - (301,)  ->  the baseline is applied to every row.
stack_corrected = stack - instrument_baseline

print("corrected stack shape:", stack_corrected.shape)
print("no loop, no copy-paste -- one expression handled 301,000 numbers.")

That single line replaces a 1000-iteration loop, and NumPy runs the work in fast
compiled code underneath. The mental model that matters: *line up the shapes, write
the operation once.*

## 4. Reductions: averaging and summarizing a batch

Reductions collapse an axis. `axis=0` collapses down the rows (giving a result per
wavelength — e.g. the **mean spectrum**); `axis=1` collapses across wavelengths
(giving a result per spectrum — e.g. each spectrum's peak height).

In [ ]:
mean_spectrum = stack_corrected.mean(axis=0)     # average over spectra -> shape (301,)
per_spectrum_peak = stack_corrected.max(axis=1)  # max of each spectrum  -> shape (1000,)

print("mean spectrum shape   :", mean_spectrum.shape, "(one value per wavelength)")
print("per-spectrum peak shape:", per_spectrum_peak.shape, "(one value per spectrum)")

# Averaging replicates is the same idea on a small slice: mean the first 5 rows.
replicates = stack_corrected[:5]
replicate_average = replicates.mean(axis=0)
print("averaged 5 replicates -> shape", replicate_average.shape,
      "| peak", round(replicate_average.max(), 3), "AU")

## 5. See the batch average

A quick look at the mean spectrum confirms the band survived correction and the
baseline is gone.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(wl, mean_spectrum)
ax.set_xlabel("wavelength (nm)")
ax.set_ylabel("absorbance (AU)")
ax.set_title("Mean of 1000 baseline-corrected spectra")
plt.tight_layout()
plt.show()

## Key Takeaways

- A spectrum is a **1-D array**; a batch is a **2-D array** (rows = spectra,
  columns = wavelengths).
- **Vectorized** arithmetic applies one operation to every element — no loops.
- **Broadcasting** lets a per-wavelength vector act on a whole stack at once.
- **Reductions** (`axis=0`, `axis=1`) average spectra or summarize each spectrum.

## Practical Checklist

- [ ] Know the **shape** of every array (`.shape`) before operating on it.
- [ ] Subtract a baseline vector from a stack instead of looping over spectra.
- [ ] Use `axis=0` to average across spectra, `axis=1` to summarize each spectrum.
- [ ] Fix a random seed whenever a step uses randomness.

## Common Mistakes

- Writing a `for` loop over spectra when one vectorized line would do.
- Confusing `axis=0` and `axis=1` — collapsing the wrong dimension.
- Shape mismatches when broadcasting (e.g. a baseline that isn't the wavelength length).

## Next Lesson

**2.2 — Indexing, Slicing, and Selecting Spectral Regions.** Whole-spectrum math is
powerful; next we focus it on the region that actually carries the chemistry.